# PaddleOCR-VL Fine-Tuning with cord-v2 Dataset

Fine-tune PaddleOCR-VL-1.5 on the naver-clova-ix/cord-v2 receipt OCR dataset using SFTTrainer.

In [ ]:
%pip install -q "transformers>=4.55.0,<5.0.0" datasets trl peft accelerate bitsandbytes Pillow

import os
os.environ["HF_HOME"] = "/dev/shm/hf_cache"

In [ ]:
import json

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault("TRACKIO_SPACE_ID", "trl-trackio")

## Configuration

In [ ]:
MODEL_NAME = "PaddlePaddle/PaddleOCR-VL-1.5"
DATASET_NAME = "naver-clova-ix/cord-v2"
OUTPUT_DIR = "PaddleOCR-VL-finetuned"
OCR_PROMPT = "Extract and parse the document content from this image."

# Training hyperparameters
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
MAX_STEPS = -1  # Set to a positive number to limit training steps (e.g., 1 for testing)

# LoRA configuration (set USE_LORA = False for full fine-tuning)
USE_LORA = True
LORA_R = 32
LORA_ALPHA = 16
LORA_TARGET_MODULES = "all-linear"

## Dataset Formatting

In [ ]:
def format_gt_parse(gt_parse: dict) -> str:
    """Convert the gt_parse dict into a readable string representation."""
    lines = []
    for key, value in gt_parse.items():
        if isinstance(value, list):
            lines.append(f"{key}:")
            for item in value:
                if isinstance(item, dict):
                    parts = [f"  {k}: {v}" for k, v in item.items() if v]
                    lines.append("\n".join(parts))
                else:
                    lines.append(f"  {item}")
        elif isinstance(value, dict):
            parts = [f"  {k}: {v}" for k, v in value.items() if v]
            if parts:
                lines.append(f"{key}:")
                lines.append("\n".join(parts))
        elif value:
            lines.append(f"{key}: {value}")
    return "\n".join(lines)


def format_cord_v2(samples: dict[str, any]) -> dict[str, list]:
    """Format cord-v2 samples into chat messages for SFTTrainer."""
    formatted_samples = {"messages": []}
    for i in range(len(samples["image"])):
        image = samples["image"][i]
        if image.mode != "RGB":
            image = image.convert("RGB")

        gt = json.loads(samples["ground_truth"][i])
        gt_parse = gt.get("gt_parse", {})
        target_text = format_gt_parse(gt_parse)

        formatted_samples["messages"].append(
            [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": OCR_PROMPT},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": target_text}],
                },
            ]
        )
    return formatted_samples

## Load Dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
dataset = dataset.map(format_cord_v2, batched=True, batch_size=32, num_proc=32)
print(dataset)

## Load Model

In [ ]:
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Training Configuration

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    max_steps=MAX_STEPS,
    logging_steps=1,
    save_strategy="epoch",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    use_cpu=not torch.cuda.is_available(),
)

peft_config = None
if USE_LORA:
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        task_type="CAUSAL_LM",
    )
    print(f"LoRA enabled: r={LORA_R}, alpha={LORA_ALPHA}")

print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    peft_config=peft_config,
)

trainer.train()

## Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")